# Offline End-to-End Tutorial Gate

This notebook is a deterministic local-data tutorial path for stabilization verification.
It demonstrates core parse/load/plot paths, workflow outputs, and an offline-safe regulatory/chip-atlas setup flow.


In [ ]:
from pathlib import Path
import json
from matplotlib import pyplot as plt

from dimelo import parse_bam, plot_depth_profile, plot_enrichment, plot_reads, workflows
from dimelo.models import SampleSpec

repo_root = Path.cwd()
data_dir = repo_root / "dimelo" / "test" / "data"
reference_dir = repo_root / "dimelo" / "test" / "output"
artifact_dir = repo_root / "artifacts" / "tutorial_offline"
artifact_dir.mkdir(parents=True, exist_ok=True)

ctcf_bam_file_updated = reference_dir / "ctcf_demo.updated.bam"
ctcf_target_regions = data_dir / "ctcf_demo_peak.bed"
ctcf_off_target_regions = data_dir / "ctcf_demo_not_peak.bed"
ref_genome_file = reference_dir / "chm13.draft_v1.0.fasta"

required_paths = [ctcf_bam_file_updated, ctcf_target_regions, ctcf_off_target_regions, ref_genome_file]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required local tutorial assets: " + ", ".join(missing))

print("Using local assets:")
for path in required_paths:
    print(f"  - {path}")


## Parse BAM (Pileup + Extract)

In [ ]:
pileup_file, pileup_regions = parse_bam.pileup(
    input_file=ctcf_bam_file_updated,
    output_name="offline_tutorial_pileup",
    ref_genome=ref_genome_file,
    output_directory=artifact_dir,
    regions=[ctcf_target_regions, ctcf_off_target_regions],
    motifs=["A,0", "CG,0"],
    thresh=190,
    window_size=1000,
    cores=1,
    quiet=True,
)

extract_file, extract_regions = parse_bam.extract(
    input_file=ctcf_bam_file_updated,
    output_name="offline_tutorial_extract",
    ref_genome=ref_genome_file,
    output_directory=artifact_dir,
    regions=[ctcf_target_regions, ctcf_off_target_regions],
    motifs=["A,0", "CG,0"],
    thresh=190,
    window_size=1000,
    cores=1,
    quiet=True,
)

print("pileup_file:", pileup_file)
print("extract_file:", extract_file)
print("pileup_regions:", pileup_regions)
print("extract_regions:", extract_regions)


## Plotting Modules (Enrichment / Depth / Reads)

In [ ]:
plt.figure(figsize=(5, 3))
plot_enrichment.by_regions(
    mod_file_name=pileup_file,
    regions_list=[ctcf_target_regions, ctcf_off_target_regions],
    motif="A,0",
    sample_names=["on-target", "off-target"],
    single_strand=False,
)
enrichment_png = artifact_dir / "enrichment_by_region.png"
plt.tight_layout()
plt.savefig(enrichment_png, dpi=150)
plt.close()

plt.figure(figsize=(6, 3))
plot_depth_profile.by_modification(
    mod_file_name=pileup_file,
    regions=ctcf_target_regions,
    motifs=["A,0", "CG,0"],
    window_size=1000,
    single_strand=False,
    smooth_window=50,
)
depth_png = artifact_dir / "depth_profile.png"
plt.tight_layout()
plt.savefig(depth_png, dpi=150)
plt.close()

single_region = "chr1:114357437-114359753"
plt.figure(figsize=(7, 3))
plot_reads.plot_reads(
    mod_file_name=extract_file,
    regions=single_region,
    motifs=["A,0", "CG,0"],
    window_size=500,
    single_strand=False,
    thresh=0.5,
    s=0.8,
)
reads_png = artifact_dir / "single_read_raster.png"
plt.tight_layout()
plt.savefig(reads_png, dpi=150)
plt.close()

print("Saved:")
print(f"  - {enrichment_png}")
print(f"  - {depth_png}")
print(f"  - {reads_png}")


## Workflow-Level Output (Shared Clustering)

In [ ]:
samples = [
    SampleSpec(
        sample_id="demo_A",
        condition="baseline",
        extract_h5=str(extract_file),
        regions_bed=str(pileup_regions),
        metadata={"pileup_path": str(pileup_file)},
    ),
    SampleSpec(
        sample_id="demo_B",
        condition="treated",
        extract_h5=str(extract_file),
        regions_bed=str(pileup_regions),
        metadata={"pileup_path": str(pileup_file)},
    ),
]

shared_result = workflows.shared_cluster_distribution(
    samples=samples,
    mode="region_anchored",
    motifs=["A,0"],
    matched_regions=pileup_regions,
    n_clusters=2,
    training_sample_per_dataset=2000,
    make_plots=False,
    quiet=True,
    cores=1,
)

shared_result.condition_distribution.head()


## Regulatory / ChIP-Atlas Setup Spec (Offline-Safe)

In [ ]:
regulatory_spec = workflows.resolve_regulatory_enrichment_spec(
    species="homo_sapiens",
    providers=["screen", "unibind"],
    target_genome="hg38",
)

regulatory_spec


In [ ]:
RUN_CHIP_ATLAS = False  # Optional live-network step

chip_atlas_kwargs = {
    "genome": "hg38",
    "mode": "per_cluster",
    "top_n_regions": 25,
    "timeout_seconds": 120.0,
}

if RUN_CHIP_ATLAS:
    chip_result = workflows.chip_atlas_cluster_enrichment_workflow(
        cluster_result=shared_result,
        **chip_atlas_kwargs,
    )
    display(chip_result)
else:
    print("Skipping live ChIP-Atlas request (RUN_CHIP_ATLAS=False).")
    print("Prepared call arguments:", chip_atlas_kwargs)


## Persist Tutorial Summary

In [ ]:
summary = {
    "pileup_file": str(pileup_file),
    "extract_file": str(extract_file),
    "pileup_regions": str(pileup_regions),
    "extract_regions": str(extract_regions),
    "plot_outputs": {
        "enrichment": str(artifact_dir / "enrichment_by_region.png"),
        "depth_profile": str(artifact_dir / "depth_profile.png"),
        "single_read_raster": str(artifact_dir / "single_read_raster.png"),
    },
    "workflow": {
        "cluster_labels": list(shared_result.model.cluster_labels),
        "condition_rows": int(shared_result.condition_distribution.shape[0]),
        "region_summary_rows": int(0 if shared_result.region_summaries is None else shared_result.region_summaries.shape[0]),
    },
    "regulatory_spec": {
        "species": regulatory_spec.species,
        "providers": list(regulatory_spec.providers),
        "target_genome": regulatory_spec.target_genome,
    },
}
summary_path = artifact_dir / "offline_tutorial_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print("Wrote", summary_path)
summary
